In [67]:
print(1)

1


In [68]:
import anndata as ad
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [69]:
import matplotlib.pyplot as plt

In [70]:
from sklearn.decomposition import PCA

In [71]:
import dilimap as dmap

In [72]:
import pickle

In [73]:
from scipy import stats

# Download_data

In [74]:
l1000_phase1_path = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated/cigs_mce/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/'
l1000_phase1_files = os.listdir(l1000_phase1_path)

In [75]:
l1000_phase1 = []
for file in tqdm(l1000_phase1_files):
    l1000_phase1.append(ad.read_h5ad(l1000_phase1_path + file))

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
100%|██████████| 2/2 [00:43<00:00, 21.93s/it]


In [76]:
dili_train_path = '../../../data/dilimap_train_val/raw/adata_training_counts.h5ad'
dili_val_path = '../../../data/dilimap_train_val/raw/adata_validation_counts.h5ad'

dili_train = ad.read_h5ad(dili_train_path)
dili_val = ad.read_h5ad(dili_val_path)

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [77]:
with open("../../dili.pkl", "rb") as file:
    df_dili = pickle.load(file)

In [78]:
df_compounds_train = dili_train.obs[['COMPOUND', 'CONCENTRATION_UM', 'TIMEPOINT_HOURS', 'SPLIT']]
df_compounds_val = dili_val.obs[['COMPOUND', 'CONCENTRATION_UM', 'TIMEPOINT_HOURS', 'SPLIT']]

In [79]:
df_compounds = pd.concat([df_compounds_train, df_compounds_val])

In [80]:
df_compounds = df_compounds.merge(df_dili[['pert_id', 'pubchem_cid']], how='left', left_on='COMPOUND', right_on='pert_id')

In [81]:
df_compounds.loc[df_compounds['COMPOUND'] == 'DMSO', 'pubchem_cid'] = 679

In [82]:
df_compounds['pubchem_cid'] = df_compounds['pubchem_cid'].fillna(-666).astype(int).astype(str).astype('category').replace({'-666': None, -666: None})

/home/icb/olga.novitskaia/tmp/ipykernel_187476/269742383.py:1: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df_compounds['pubchem_cid'] = df_compounds['pubchem_cid'].fillna(-666).astype(int).astype(str).astype('category').replace({'-666': None, -666: None})


# Overlapping compounds

In [83]:
DILI_TIME = 24.0  
overlapping_compounds1 = {}
for j, l1000_phase1_j in enumerate(l1000_phase1):
    l1000_24h = l1000_phase1_j.obs[l1000_phase1_j.obs['pert_time_h'] == DILI_TIME]
    overlapping_compounds1[j] = len(set(df_compounds['pubchem_cid']).intersection(set(l1000_24h['pubchem_cid'])))

In [84]:
sorted(overlapping_compounds1.items(), key=lambda item: item[1], reverse=True)[:15]

[(1, 175), (0, 174)]

In [85]:
def match_by_cid_and_closest_log_dose(dili, l1000_adata):
    l1000_obs = l1000_adata.obs.copy()
    l1000_obs['log_dose'] = np.log(l1000_obs['pert_dose_uM'])

    dili['log_dose'] = np.log(dili['CONCENTRATION_UM'])

    l1000_by_cid = {cid: grp for cid, grp in l1000_obs.groupby('pubchem_cid', observed=True)}

    matched_l1000_idx = []
    difference = []
    for _, row in dili.iterrows():
        l1000_grp = l1000_by_cid[row['pubchem_cid']]
        closest = (l1000_grp['log_dose'] - row['log_dose']).abs().idxmin()
        difference.append((l1000_grp['log_dose'] - row['log_dose']).abs().min())
        matched_l1000_idx.append(closest)

    dili['matched_cigs_idx'] = matched_l1000_idx
    dili['diff'] = difference
    return dili, l1000_obs

def return_embeddings(adata, dim=64,):
    adata_obs = adata.obs.copy()
    pca = PCA(n_components=dim, random_state=42)
    arr = adata.layers['logFC']
    mask = ~np.isnan(arr).any(axis=0)
    arr_clean = arr[:, mask]
    print(f'logFC before {arr.shape} , after {arr_clean.shape}')
    emb_logFC = pca.fit_transform(arr_clean)
    
    pca = PCA(n_components=dim, random_state=42)
    arr = adata.layers['t']
    mask = ~np.isnan(arr).any(axis=0)
    arr_clean = arr[:, mask]
    print(f't before {arr.shape} , after {arr_clean.shape}')
    emb_t = pca.fit_transform(arr_clean)
    
    adata_obs['PCA.logFC'] = emb_logFC.tolist()
    adata_obs['PCA.t'] = emb_t.tolist()
    return adata_obs

In [86]:
to_check = sorted(overlapping_compounds1.items(), key=lambda item: item[1], reverse=True)[:15]
for item in to_check:
    l_id = item[0]
    compounds_overlapped = list(set(df_compounds['pubchem_cid']).intersection(l1000_phase1[l_id][l1000_phase1[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))
    df_compounds_filtered = df_compounds[df_compounds['pubchem_cid'].isin(compounds_overlapped)].copy().dropna()

    #df_compounds_filtered = df_compounds_filtered[(df_compounds_filtered['CONCENTRATION_UM'] < 15) & (df_compounds_filtered['CONCENTRATION_UM'] > 5)].copy()
    
    
    l1000_phase1_no_duplicates = l1000_phase1[l_id][~l1000_phase1[l_id].obs.index.duplicated()]
    l1000_phase1_filtered = l1000_phase1_no_duplicates[l1000_phase1_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
    l1000_phase1_filtered_time = l1000_phase1_filtered[l1000_phase1_filtered.obs['pert_time_h'] == 24].copy()

    
    df_compounds_filtered_, _ = match_by_cid_and_closest_log_dose(df_compounds_filtered, l1000_phase1_filtered_time)
    l1000_phase1_filtered_time_obs = l1000_phase1_filtered_time.obs
    df_compounds_filtered_ = df_compounds_filtered_.merge(l1000_phase1_filtered_time_obs[['pert_dose_uM', 'pert_time_h']], how='left', left_on='matched_cigs_idx', right_index=True)

    df_compounds_filtered_ = df_compounds_filtered_.sort_values('diff').drop_duplicates(subset='COMPOUND')
    df_compounds_filtered_['LOG_CONCENTRATION_UM'] = np.log(df_compounds_filtered_['CONCENTRATION_UM'])
    
    median = np.median(df_compounds_filtered_['LOG_CONCENTRATION_UM'].values)
    mad = stats.median_abs_deviation(df_compounds_filtered_['LOG_CONCENTRATION_UM'].values)
    df_compounds_filtered_ = df_compounds_filtered_[(df_compounds_filtered_['LOG_CONCENTRATION_UM'] <= median + mad * 2.5) \
                                & (df_compounds_filtered_['LOG_CONCENTRATION_UM'] >= median - mad * 2.5)].reset_index(drop=True)
    
    print(l_id, 
          len(df_compounds_filtered_['COMPOUND'].unique()), 
          len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()))
    

1 166 166
0 165 165


# Processing

In [87]:
cols_l1000 = ['PCA.logFC', 
              'PCA.t',
              'cell_type',
              'pert_dose_uM', 
              'pert_time_h']

In [88]:
rename_l1000 = {'cell_type': 'cigs_mce_cell_type',
                'pert_time_h': 'cigs_mce_pert_time_h',
                'pert_dose_uM': 'cigs_mce_pert_dose_uM'}

## Phase 1

In [89]:
def construct_data_filtered(query, reference, l_id):
    
    df_compounds = query.copy()
    compounds_overlapped = list(set(df_compounds['pubchem_cid']).intersection(reference[l_id][reference[l_id].obs['pert_time_h'] == 24].obs['pubchem_cid']))
    df_compounds_filtered = df_compounds[df_compounds['pubchem_cid'].isin(compounds_overlapped)].copy().dropna()
    #df_compounds_filtered = df_compounds_filtered[(df_compounds_filtered['CONCENTRATION_UM'] < 15) & (df_compounds_filtered['CONCENTRATION_UM'] > 5)].copy()
    
    reference_no_duplicates = reference[l_id][~reference[l_id].obs.index.duplicated()].copy()
    reference_filtered = reference_no_duplicates[reference_no_duplicates.obs['pubchem_cid'].isin(compounds_overlapped)]
    reference_filtered_time = reference_filtered[reference_filtered.obs['pert_time_h'] == 24].copy()
    
    df_compounds_filtered_, _ = match_by_cid_and_closest_log_dose(df_compounds_filtered, reference_filtered_time)
    
    ## filter by prevailing dose
    reference_filtered_time_obs = reference_filtered_time.obs
    
    df_compounds_filtered_ = df_compounds_filtered_ = df_compounds_filtered_.merge(reference_filtered_time_obs[['pert_dose_uM', 'pert_time_h']], how='left', left_on='matched_cigs_idx', right_index=True)

    df_compounds_filtered_ = df_compounds_filtered_.sort_values('diff').drop_duplicates(subset='COMPOUND')
    df_compounds_filtered_['LOG_CONCENTRATION_UM'] = np.log(df_compounds_filtered_['CONCENTRATION_UM'])
    
    median = np.median(df_compounds_filtered_['LOG_CONCENTRATION_UM'].values)
    mad = stats.median_abs_deviation(df_compounds_filtered_['LOG_CONCENTRATION_UM'].values)
    df_compounds_filtered_ = df_compounds_filtered_[(df_compounds_filtered_['LOG_CONCENTRATION_UM'] <= median + mad * 2.5) \
                                & (df_compounds_filtered_['LOG_CONCENTRATION_UM'] >= median - mad * 2.5)].reset_index(drop=True)
    
    
    df_compounds_filtered_ = df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]
    #df_compounds_filtered_ = df_compounds_filtered_.drop_duplicates(['COMPOUND', 'CONCENTRATION_UM'])
    if not df_compounds_filtered_[['COMPOUND', 'CONCENTRATION_UM']].shape[0] == df_compounds_filtered_[['COMPOUND']].shape[0]:
        print(l_id)

    return df_compounds_filtered_, reference_no_duplicates, reference_filtered, reference_filtered_time

In [90]:
def get_v1(df_compounds_filtered_, reference_filtered_time, dim=64):
    #V1
    
    df = reference_filtered_time[(reference_filtered_time.obs.index.isin(df_compounds_filtered_['matched_cigs_idx']))]
    df_emb = return_embeddings(df, dim)
    
    print('matched doses:', df_emb['pert_dose_uM'].unique())
    df_emb_dili = df_compounds_filtered_.merge(df_emb[cols_l1000].rename(columns=rename_l1000), left_on='matched_cigs_idx', right_index=True, how='left')
    df_emb_dili['version'] = 'v1'
    df_emb_dili['dim'] = dim
    return df_emb_dili

In [91]:
def get_v2(df_compounds_filtered_, reference_no_duplicates, dim=64):
    #V2
    df = reference_no_duplicates.copy()
    df_emb = return_embeddings(df, dim)
    
    df_emb_dili = df_compounds_filtered_.merge(df_emb[cols_l1000].rename(columns=rename_l1000), left_on='matched_cigs_idx', right_index=True, how='left')
    df_emb_dili['version'] = 'v2'
    df_emb_dili['dim'] = dim
    return df_emb_dili

In [92]:
def get_v3(df_compounds_filtered_, reference_no_duplicates, dim=64):
    df = reference_no_duplicates[(reference_no_duplicates.obs['pert_dose_uM'].isin([10]))&(reference_no_duplicates.obs['pert_time_h'].isin([24.]))].copy()
    df_emb = return_embeddings(df, dim)
    df_emb_dili = df_compounds_filtered_.merge(df_emb[cols_l1000].rename(columns=rename_l1000), left_on='matched_cigs_idx', right_index=True, how='left')
    df_emb_dili['version'] = 'v3'
    df_emb_dili['dim'] = dim
    return df_emb_dili

In [93]:
def get_v4(df_compounds_filtered_, reference_filtered, dim=64):
    df = reference_filtered.copy()
    df_emb = return_embeddings(df, dim)
    df_emb_dili = df_compounds_filtered_.merge(df_emb[cols_l1000].rename(columns=rename_l1000), left_on='matched_cigs_idx', right_index=True, how='left')
    df_emb_dili['version'] = 'v4'
    df_emb_dili['dim'] = dim
    return df_emb_dili

## Phase 1

In [94]:
columns = ['COMPOUND', 
       'CONCENTRATION_UM', 'TIMEPOINT_HOURS',
        'SPLIT',
        'pubchem_cid', 'matched_cigs_idx',
        'cigs_mce_cell_type',
        'cigs_mce_pert_dose_uM', 'cigs_mce_pert_time_h', 
        'PCA.logFC', 'PCA.t', 'version', 'dim',
       ]

In [95]:
os.makedirs("cigs_not_strict_filtration", exist_ok=True)

In [96]:
to_check = sorted(overlapping_compounds1.items(), key=lambda item: item[1], reverse=True)[:15]

for item in tqdm(to_check):
    embeddings_phase1 = []
    l_id = item[0]
    df_compounds_filtered_, reference_no_duplicates, reference_filtered, reference_filtered_time = construct_data_filtered(df_compounds, l1000_phase1, l_id)
    if len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()) >= 50:
        print(len(df_compounds_filtered_[df_compounds_filtered_['pert_dose_uM'] == 10]['COMPOUND'].unique()))
        for dim in [32, 64, 128]:
            try:
                embeddings_phase1.append(get_v1(df_compounds_filtered_, reference_filtered_time, dim=dim))
            except:
                pass
            try:
                embeddings_phase1.append(get_v2(df_compounds_filtered_, reference_no_duplicates, dim=dim))
            except:
                pass
            try:    
                embeddings_phase1.append(get_v3(df_compounds_filtered_, reference_no_duplicates, dim=dim))
            except:
                pass
            try:
                embeddings_phase1.append(get_v4(df_compounds_filtered_, reference_filtered, dim=dim))
            except:
                pass
        dili_emb = pd.concat(embeddings_phase1)
        dili_emb = dili_emb[~dili_emb['PCA.t'].isna()]
        dili_emb = dili_emb[columns].reset_index(drop=True).copy()
        
        ct = l1000_phase1[l_id].obs['cell_type'].iloc[0]
        
        dili_emb.to_pickle('./cigs_not_strict_filtration/dili_PCA_emb_'+ ct +'.pkl')

  0%|          | 0/2 [00:00<?, ?it/s]

166
logFC before (166, 2881) , after (166, 1939)
t before (166, 2881) , after (166, 1939)
matched doses: [10.]
logFC before (10869, 2881) , after (10869, 1939)
t before (10869, 2881) , after (10869, 1939)
logFC before (10869, 2881) , after (10869, 1939)
t before (10869, 2881) , after (10869, 1939)
logFC before (229, 2881) , after (229, 1939)
t before (229, 2881) , after (229, 1939)
logFC before (166, 2881) , after (166, 1939)
t before (166, 2881) , after (166, 1939)
matched doses: [10.]
logFC before (10869, 2881) , after (10869, 1939)
t before (10869, 2881) , after (10869, 1939)
logFC before (10869, 2881) , after (10869, 1939)
t before (10869, 2881) , after (10869, 1939)
logFC before (229, 2881) , after (229, 1939)
t before (229, 2881) , after (229, 1939)
logFC before (166, 2881) , after (166, 1939)
t before (166, 2881) , after (166, 1939)
matched doses: [10.]
logFC before (10869, 2881) , after (10869, 1939)
t before (10869, 2881) , after (10869, 1939)
logFC before (10869, 2881) , afte

 50%|█████     | 1/2 [01:40<01:40, 100.81s/it]

165
logFC before (165, 2345) , after (165, 1544)
t before (165, 2345) , after (165, 1544)
matched doses: [10.]
logFC before (11127, 2345) , after (11127, 1544)
t before (11127, 2345) , after (11127, 1544)
logFC before (11127, 2345) , after (11127, 1544)
t before (11127, 2345) , after (11127, 1544)
logFC before (228, 2345) , after (228, 1544)
t before (228, 2345) , after (228, 1544)
logFC before (165, 2345) , after (165, 1544)
t before (165, 2345) , after (165, 1544)
matched doses: [10.]
logFC before (11127, 2345) , after (11127, 1544)
t before (11127, 2345) , after (11127, 1544)
logFC before (11127, 2345) , after (11127, 1544)
t before (11127, 2345) , after (11127, 1544)
logFC before (228, 2345) , after (228, 1544)
t before (228, 2345) , after (228, 1544)
logFC before (165, 2345) , after (165, 1544)
t before (165, 2345) , after (165, 1544)
matched doses: [10.]
logFC before (11127, 2345) , after (11127, 1544)
t before (11127, 2345) , after (11127, 1544)
logFC before (11127, 2345) , afte

100%|██████████| 2/2 [05:05<00:00, 152.90s/it]
